# OE2B-07 — Evaluación cross-country (UrBAN 2021)

## ¿Qué hace este notebook?

Evalúa si los modelos xnorC entrenados con datos de EE.UU. (Yang + NU-HIVE)
generalizan a colmenas de **Francia** (UrBAN 2021).

Esta es la prueba más exigente del pipeline: el modelo nunca vio colmenas europeas,
ni el equipamiento de grabación francés, ni las condiciones climáticas de Francia.
Si xnorC funciona acá, significa que aprendió algo sobre el comportamiento acústico
de las abejas, no simplemente las características del equipo o el entorno de grabación.

### Estrategia

1. Cargamos el inventario de UrBAN 2021 y extraemos features con las mismas funciones de OE2B-04
2. Construimos pares de test siguiendo las mismas reglas de OE2B-03
   (sin R4/R5 — usamos todos los archivos como el caso controlado NU-HIVE)
3. Para cada experimento (B1–B4), evaluamos el mejor modelo de OE2B-05
   (el fold con mayor AUC en k-hive-fold)
4. Comparamos los resultados con el k-hive-fold

### Inputs
- `../data/processed/urban/urban_inventario.csv` — inventario de UrBAN 2021
- `../data/processed/modelos/xnorC_B{n}_fold{k}.pt` — modelos de OE2B-05

### Outputs
- `../data/processed/resultados/evaluacion_cross_country.csv`
- `../data/processed/resultados/grafico_cross_country.png`

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import librosa
import soundfile as sf

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

print("Importaciones listas")

In [ ]:
# Rutas de entrada
RUTA_INV_URBAN   = Path('../data/processed/urban/urban_inventario.csv')
CARPETA_FEATURES_KHIVE = Path('../data/processed/features')  # features de Yang+NU-HIVE (para el índice)
CARPETA_MODELOS  = Path('../data/processed/modelos')
CARPETA_RESULTADOS = Path('../data/processed/resultados')
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Parámetros de preprocesamiento (iguales que OE2B-04)
SR_OBJETIVO  = 8000
DURACION_SEG = 60
N_MUESTRAS   = SR_OBJETIVO * DURACION_SEG

# Parámetros de features (iguales que OE2B-04)
N_FFT_MEL = 512;  HOP_LENGTH_MEL = 256;  N_MELS = 64
N_MFCC    = 20;   N_FFT_MFCC = 512;      HOP_LENGTH_MFCC = 256
N_FFT_BIN = 256;  HOP_LENGTH_BIN = 128
N_BINS_CQT = 72;  BINS_PER_OCT_CQT = 12; FMIN_CQT = 32.7; HOP_LENGTH_CQT = 256

DEVICE     = torch.device('cpu')
BATCH_SIZE = 64
SEMILLA    = 42

EXPERIMENTOS = {
    'B1': 'features_B1_mel.npy',
    'B2': 'features_B2_mfcc.npy',
    'B3': 'features_B3_fft.npy',
    'B4': 'features_B4_cqt.npy',
}

print("Configuración lista")
print(f"Inventario UrBAN: {RUTA_INV_URBAN} — existe: {RUTA_INV_URBAN.exists()}")

## 2. Cargar inventario UrBAN 2021

El inventario de UrBAN 2021 debe tener las mismas columnas que los otros inventarios:
`ruta`, `estado` (QR/QL), `colmena_id`, `fecha`.

Si el inventario aún no existe, esta celda lo indica claramente.

In [ ]:
if not RUTA_INV_URBAN.exists():
    print("⚠️  No se encontró el inventario de UrBAN 2021.")
    print(f"   Ruta esperada: {RUTA_INV_URBAN}")
    print()
    print("   Para generarlo, crear un notebook OE2B-01b-auditoria-urban.ipynb")
    print("   siguiendo la misma estructura que OE2B-02 (Yang).")
    print("   Columnas necesarias: ruta, estado, colmena_id, fecha")
    df_urban = None
else:
    df_urban = pd.read_csv(RUTA_INV_URBAN)
    # Solo archivos con etiqueta
    df_urban = df_urban[df_urban['estado'].isin(['QR', 'QL'])].copy()
    df_urban['dataset'] = 'urban'
    df_urban['es_controlado'] = True  # tratamos UrBAN como controlado (sin filtro de distancia)

    print("Inventario UrBAN 2021:")
    print(df_urban.groupby(['colmena_id', 'estado']).size().unstack(fill_value=0))
    print(f"\nTotal archivos: {len(df_urban):,}")

## 3. Extraer features de UrBAN 2021

Usamos exactamente las mismas funciones de extracción que OE2B-04.
Esto es fundamental: si cambiamos los parámetros, los features no son comparables
con los que el modelo vio durante el entrenamiento.

In [ ]:
# Funciones de preprocesamiento y extracción (iguales a OE2B-04)

def cargar_audio(ruta, sr_objetivo=SR_OBJETIVO, n_muestras=N_MUESTRAS):
    """Carga, convierte a mono, resamplea, normaliza y ajusta duración."""
    try:
        audio, sr_orig = sf.read(ruta, always_2d=False)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        audio = audio.astype(np.float32)
        if sr_orig != sr_objetivo:
            audio = librosa.resample(audio, orig_sr=sr_orig, target_sr=sr_objetivo)
        max_amp = np.abs(audio).max()
        if max_amp > 0:
            audio = audio / max_amp
        if len(audio) >= n_muestras:
            audio = audio[:n_muestras]
        else:
            audio = np.pad(audio, (0, n_muestras - len(audio)))
        return audio
    except Exception as e:
        print(f"⚠️  Error cargando {ruta}: {e}")
        return None

def extraer_B1_mel(audio):
    S = librosa.feature.melspectrogram(y=audio, sr=SR_OBJETIVO, n_fft=N_FFT_MEL,
                                        hop_length=HOP_LENGTH_MEL, n_mels=N_MELS)
    return librosa.power_to_db(S, ref=np.max).mean(axis=1).astype(np.float32)

def extraer_B2_mfcc(audio):
    mfcc  = librosa.feature.mfcc(y=audio, sr=SR_OBJETIVO, n_mfcc=N_MFCC,
                                   n_fft=N_FFT_MFCC, hop_length=HOP_LENGTH_MFCC)
    delta = librosa.feature.delta(mfcc)
    return np.concatenate([mfcc.mean(axis=1), delta.mean(axis=1)]).astype(np.float32)

def extraer_B3_fft(audio):
    S = np.abs(librosa.stft(audio, n_fft=N_FFT_BIN, hop_length=HOP_LENGTH_BIN))
    return S.mean(axis=1).astype(np.float32)

def extraer_B4_cqt(audio):
    C = np.abs(librosa.cqt(audio, sr=SR_OBJETIVO, hop_length=HOP_LENGTH_CQT,
                            fmin=FMIN_CQT, n_bins=N_BINS_CQT,
                            bins_per_octave=BINS_PER_OCT_CQT))
    return C.mean(axis=1).astype(np.float32)

FUNCIONES_FEATURES = {
    'B1': extraer_B1_mel,
    'B2': extraer_B2_mfcc,
    'B3': extraer_B3_fft,
    'B4': extraer_B4_cqt,
}

print("Funciones de extracción definidas")

In [ ]:
if df_urban is None:
    print("Saltando extracción: inventario UrBAN no disponible")
    features_urban = {}
    archivo_a_indice_urban = {}
else:
    from tqdm.notebook import tqdm

    print(f"Extrayendo features de {len(df_urban)} archivos UrBAN...")
    n = len(df_urban)
    matrices_urban = {exp: [None] * n for exp in EXPERIMENTOS}

    for i, (_, fila) in enumerate(tqdm(df_urban.iterrows(), total=n)):
        audio = cargar_audio(fila['ruta'])
        if audio is None:
            for exp, fn in FUNCIONES_FEATURES.items():
                vec_prueba = fn(np.zeros(N_MUESTRAS, dtype=np.float32))
                matrices_urban[exp][i] = np.zeros_like(vec_prueba)
            continue
        for exp, fn in FUNCIONES_FEATURES.items():
            try:
                matrices_urban[exp][i] = fn(audio)
            except Exception as e:
                print(f"⚠️  {exp} error en {fila['ruta']}: {e}")
                vec_prueba = fn(np.zeros(N_MUESTRAS, dtype=np.float32))
                matrices_urban[exp][i] = np.zeros_like(vec_prueba)

    features_urban = {exp: np.array(vecs) for exp, vecs in matrices_urban.items()}
    archivo_a_indice_urban = {
        Path(row['ruta']).name: idx
        for idx, row in df_urban.reset_index(drop=True).iterrows()
    }

    print("Shapes:")
    for exp, arr in features_urban.items():
        print(f"  {exp}: {arr.shape}")

## 4. Construir pares de test para UrBAN

Construimos pares cross-colmena: comparamos segmentos de distintas colmenas de UrBAN.
No aplicamos R4/R5 — usamos todos los archivos disponibles (como NU-HIVE controlado).

**Importante:** estos pares son solo para test, nunca para entrenar.

In [ ]:
if df_urban is None:
    print("Saltando: inventario UrBAN no disponible")
    df_pares_urban = None
else:
    rng = np.random.default_rng(SEMILLA)
    MAX_PARES_POR_COLMENA = 200  # limitamos para no tardar demasiado
    pares = []

    for colmena in sorted(df_urban['colmena_id'].unique()):
        df_col = df_urban[df_urban['colmena_id'] == colmena]
        cand_qr = df_col[df_col['estado'] == 'QR']
        cand_ql = df_col[df_col['estado'] == 'QL']

        n_dif = min(MAX_PARES_POR_COLMENA, len(cand_qr), len(cand_ql))
        if n_dif == 0:
            print(f"  ⚠️  Colmena {colmena}: sin pares posibles (QR={len(cand_qr)}, QL={len(cand_ql)})")
            continue

        idx_qr = rng.choice(len(cand_qr), size=n_dif, replace=False)
        idx_ql = rng.choice(len(cand_ql), size=n_dif, replace=False)
        sel_qr = cand_qr.iloc[idx_qr].reset_index(drop=True)
        sel_ql = cand_ql.iloc[idx_ql].reset_index(drop=True)

        # Pares DIFERENTE
        for i in range(n_dif):
            pares.append({'ruta_a': sel_qr.loc[i, 'ruta'], 'estado_a': 'QR',
                          'ruta_b': sel_ql.loc[i, 'ruta'], 'estado_b': 'QL',
                          'etiqueta': 0, 'colmena': colmena})

        # Pares IGUAL QR+QR
        n_ig = min(n_dif // 2, len(cand_qr) // 2)
        if n_ig >= 1:
            idx = rng.choice(len(cand_qr), size=n_ig * 2, replace=False)
            sa  = cand_qr.iloc[idx[:n_ig]].reset_index(drop=True)
            sb  = cand_qr.iloc[idx[n_ig:]].reset_index(drop=True)
            for i in range(n_ig):
                pares.append({'ruta_a': sa.loc[i, 'ruta'], 'estado_a': 'QR',
                              'ruta_b': sb.loc[i, 'ruta'], 'estado_b': 'QR',
                              'etiqueta': 1, 'colmena': colmena})

        # Pares IGUAL QL+QL
        n_ig_ql = n_dif - n_ig
        if n_ig_ql >= 1 and len(cand_ql) >= 2:
            n_ig_ql = min(n_ig_ql, len(cand_ql) // 2)
            idx = rng.choice(len(cand_ql), size=n_ig_ql * 2, replace=False)
            sa  = cand_ql.iloc[idx[:n_ig_ql]].reset_index(drop=True)
            sb  = cand_ql.iloc[idx[n_ig_ql:]].reset_index(drop=True)
            for i in range(n_ig_ql):
                pares.append({'ruta_a': sa.loc[i, 'ruta'], 'estado_a': 'QL',
                              'ruta_b': sb.loc[i, 'ruta'], 'estado_b': 'QL',
                              'etiqueta': 1, 'colmena': colmena})

        n_col = sum(1 for p in pares if p['colmena'] == colmena)
        print(f"  Colmena {colmena}: {n_col} pares")

    df_pares_urban = pd.DataFrame(pares)
    print(f"\nTotal pares UrBAN: {len(df_pares_urban):,}")
    print(f"  DIFERENTE: {(df_pares_urban['etiqueta']==0).sum():,}")
    print(f"  IGUAL:     {(df_pares_urban['etiqueta']==1).sum():,}")

## 5. Evaluar modelos sobre UrBAN

Para cada experimento usamos el modelo del fold con mejor AUC en k-hive-fold.
Si no hay resultados previos del k-hive-fold, usamos el fold 0 por defecto.

In [ ]:
# Definición de la red (igual que OE2B-05 y OE2B-06)
class Encoder(nn.Module):
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dim_entrada, dim_embedding*2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim_embedding*2, dim_embedding), nn.ReLU()
        )
    def forward(self, x): return self.red(x)

class xnorC(nn.Module):
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        self.encoder = Encoder(dim_entrada, dim_embedding, dropout)
        self.clf = nn.Sequential(
            nn.Linear(dim_embedding, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 2)
        )
    def forward(self, a, b):
        return self.clf(torch.abs(self.encoder(a) - self.encoder(b)))


class DatasetPares(Dataset):
    def __init__(self, df_pares, features_array, mapa):
        self.pares = df_pares[
            df_pares.apply(lambda f:
                Path(f['ruta_a']).name in mapa and
                Path(f['ruta_b']).name in mapa, axis=1)
        ].reset_index(drop=True)
        self.features = features_array
        self.mapa = mapa
    def __len__(self): return len(self.pares)
    def __getitem__(self, idx):
        f = self.pares.iloc[idx]
        return (
            torch.tensor(self.features[self.mapa[Path(f['ruta_a']).name]], dtype=torch.float32),
            torch.tensor(self.features[self.mapa[Path(f['ruta_b']).name]], dtype=torch.float32),
            torch.tensor(int(f['etiqueta']), dtype=torch.long)
        )

print("Arquitectura definida")

In [ ]:
if df_pares_urban is None or df_pares_urban.empty:
    print("Saltando evaluación: sin pares UrBAN")
else:
    # Intentar leer el CSV del k-hive-fold para elegir el mejor fold
    ruta_khive = CARPETA_RESULTADOS / 'evaluacion_khivefold.csv'
    if ruta_khive.exists():
        df_khive = pd.read_csv(ruta_khive)
        mejor_fold_por_exp = (
            df_khive.loc[df_khive.groupby('experimento')['auc'].idxmax()]
            .set_index('experimento')['fold']
            .to_dict()
        )
        print("Mejor fold por experimento (según AUC en k-hive-fold):")
        for exp, fold in mejor_fold_por_exp.items():
            print(f"  {exp}: fold {fold}")
    else:
        print("⚠️  No se encontró evaluacion_khivefold.csv — usando fold 0 para todos")
        mejor_fold_por_exp = {exp: 0 for exp in EXPERIMENTOS}

    resultados_urban = []

    for exp in EXPERIMENTOS:
        if exp not in features_urban or len(features_urban[exp]) == 0:
            print(f"\n⚠️  {exp}: sin features UrBAN")
            continue

        fold_elegido = mejor_fold_por_exp.get(exp, 0)
        ruta_modelo  = CARPETA_MODELOS / f'xnorC_{exp}_fold{fold_elegido}.pt'

        if not ruta_modelo.exists():
            print(f"\n⚠️  {exp}: modelo no encontrado en {ruta_modelo}")
            continue

        print(f"\n{exp} — modelo del fold {fold_elegido}")

        # Cargar modelo
        ckpt   = torch.load(ruta_modelo, map_location='cpu')
        modelo = xnorC(ckpt['dim_entrada'], ckpt['embedding_dim'], ckpt['dropout'])
        modelo.load_state_dict(ckpt['estado_modelo'])
        modelo.eval()

        # Dataset y loader
        ds     = DatasetPares(df_pares_urban, features_urban[exp], archivo_a_indice_urban)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)

        # Inferencia
        y_true, y_pred, y_prob = [], [], []
        with torch.no_grad():
            for a, b, etiq in loader:
                logits = modelo(a, b)
                probs  = torch.softmax(logits, dim=1)
                y_true.extend(etiq.numpy())
                y_pred.extend(torch.argmax(logits, dim=1).numpy())
                y_prob.extend(probs[:, 1].numpy())

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        y_prob = np.array(y_prob)

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average='weighted')
        auc = roc_auc_score(y_true, y_prob)

        print(f"  acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  (n={len(y_true)})")

        resultados_urban.append({
            'experimento': exp,
            'fold_usado': fold_elegido,
            'n_pares': len(y_true),
            'accuracy': round(acc, 4),
            'f1': round(f1, 4),
            'auc': round(auc, 4),
        })

    df_urban_res = pd.DataFrame(resultados_urban)
    df_urban_res.to_csv(CARPETA_RESULTADOS / 'evaluacion_cross_country.csv', index=False)
    print(f"\nGuardado: {CARPETA_RESULTADOS / 'evaluacion_cross_country.csv'}")

## 6. Comparación k-hive-fold vs cross-country

In [ ]:
ruta_khive  = CARPETA_RESULTADOS / 'evaluacion_khivefold.csv'
ruta_urban  = CARPETA_RESULTADOS / 'evaluacion_cross_country.csv'

if not ruta_khive.exists() or not ruta_urban.exists():
    print("Falta alguno de los dos CSVs para comparar.")
else:
    df_k = pd.read_csv(ruta_khive)
    df_u = pd.read_csv(ruta_urban)

    # Promedio k-hive-fold por experimento
    prom_khive = df_k.groupby('experimento')[['accuracy', 'f1', 'auc']].mean().round(3)
    prom_khive.columns = [f'{c}_khive' for c in prom_khive.columns]

    # Resultados cross-country
    prom_urban = df_u.set_index('experimento')[['accuracy', 'f1', 'auc']].round(3)
    prom_urban.columns = [f'{c}_urban' for c in prom_urban.columns]

    comparacion = prom_khive.join(prom_urban)
    print("Comparación k-hive-fold (promedio) vs UrBAN 2021 (cross-country):")
    print(comparacion.to_string())

    # Gráfico de barras comparativo
    exps = comparacion.index.tolist()
    x    = np.arange(len(exps))
    ancho = 0.35

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, metrica in zip(axes, ['accuracy', 'f1', 'auc']):
        vals_k = comparacion[f'{metrica}_khive'].values
        vals_u = comparacion[f'{metrica}_urban'].values
        ax.bar(x - ancho/2, vals_k, ancho, label='k-hive-fold', color='#1565C0', alpha=0.8)
        ax.bar(x + ancho/2, vals_u, ancho, label='UrBAN 2021', color='#EF6C00', alpha=0.8)
        ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='azar')
        ax.set_xticks(x)
        ax.set_xticklabels(exps)
        ax.set_title(metrica.upper())
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('xnorC — k-hive-fold (EE.UU.) vs cross-country (Francia UrBAN 2021)', fontsize=12)
    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / 'grafico_cross_country.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nGuardado: {CARPETA_RESULTADOS / 'grafico_cross_country.png'}")

## Notas sobre la evaluación cross-country

### Qué indica una caída de rendimiento

Es normal y esperable que el AUC en UrBAN sea menor que en k-hive-fold:
- El modelo fue entrenado con audio californiano y neozelandés, no europeo
- El equipamiento de grabación de UrBAN es distinto
- Las condiciones climáticas y de manejo de colmenas son distintas

Una caída moderada (ej: AUC 0.75 → 0.65) es una señal positiva:
indica que xnorC aprendió algo genuino sobre el comportamiento acústico
de las abejas, aunque no generalice perfectamente.

### Qué indica que el modelo no aprende nada

Un AUC cercano a 0.5 en ambas evaluaciones indica que las representaciones
de features (con promedio temporal) no capturan suficiente información
discriminativa. En ese caso, aplicar las mejoras del OE2B-04 (sección 8)
sería el siguiente paso.